# Embedding APIs

## 1. First: two different meanings of "embedding"

The word **embedding** is used for two related but different things.

### LLM token embedding

Inside an LLM:

```text
text
↓
tokenizer
↓
token IDs
↓
embedding lookup
↓
one vector per token
```

For example:

```text
"cat" → token ID 1234 → [0.2, -0.7, 1.1, ...]
```

The token ID answers:

> **Which token is this?**

The embedding answers:

> **How should the model initially represent this token?**

An important detail:

The initial token embedding is **not yet contextual**.

For example, `bank` initially gets the same token embedding in:

```text
I deposited money at the bank.
```

and:

```text
I sat on the river bank.
```

After the Transformer processes the sequence, each token gets a **contextual representation**.

So the richer picture is:

```text
token IDs
↓
initial token embeddings
↓
Transformer
↓
contextual representation for every token
```


Modern LLMs commonly use thousands of numbers per token.

---

## 2. Embedding API is a different product

An Embedding API usually does this:

```text
variable-length text
↓
embedding model
↓
one fixed-size vector
```

For example:

```text
"cat"
↓
[... 1024 numbers ...]
```

and:

```text
"The cat is sleeping on the sofa."
↓
[... 1024 numbers ...]
```

and even:

```text
a long paragraph
↓
[... 1024 numbers ...]
```

So a useful mental model is:

> **Embedding API = fixed-size summary representation**

Even better:

> **Embedding API = lossy semantic compression**

It intentionally throws away information.

---

# 3. What problem is the Embedding API trying to solve?

Suppose we have 10 million documents.

A user asks:

```text
How do I reset my password?
```

We want to find documents with similar meaning:

```text
I forgot my login credentials.
How can I change my password?
Recovering access to your account.
```

The naive solution would be to ask a large LLM to deeply compare the query against every document.

That would be extremely expensive.

Instead, convert everything into vectors ahead of time:

```text
document 1 → vector 1
document 2 → vector 2
document 3 → vector 3
...
```

Then convert the query:

```text
query → query vector
```

Now retrieval becomes approximately:

find vectors close to query vector using cosine similarity, dot product, or a related metric.

The original language problem has been transformed into a geometry/search problem.

---

# 4. Why fixed-size vectors?

This is mainly an engineering decision.

If every document becomes: a fixed size vector

then every document has the same compact representation.

That makes it possible to:

- store millions or billions of vectors
- build efficient indexes
- compare vectors cheaply
- use approximate nearest-neighbor search
- retrieve candidates before spending expensive LLM compute

So the dominant architecture is:

```text
rich variable-length information
↓
compress
↓
small fixed-size representation
↓
cheap search
```

This is not the **ONLY** mathematically possible approach.

It is a very successful engineering **trade-off**.

---

# 5. What gets lost?

Suppose the original LLM representation looks conceptually like:

For example:

```text
1000 tokens
×
4000 numbers/token
≈
4,000,000 numbers
```

A text embedding might compress the whole thing into:

```text
1024 numbers
```

That is enormous compression.

Therefore an Embedding API is **NOT** trying to preserve everything.

It may preserve useful semantic information such as:

```text
topic
meaning
intent
semantic similarity
```

while discarding much of:

```text
exact wording
exact token order
punctuation
fine-grained details
many individual facts
```

You generally cannot reconstruct the original text from its embedding.

---

# 6. Does larger embedding dimension mean higher quality?

Not automatically.

Within the **same model**, if the model supports different output dimensions:

```text
256
512
1024
2048
```

then more dimensions generally give the representation more capacity, while costing more storage and compute.

A well-trained 256-dimensional embedding model can outperform an older 1536-dimensional model.

Quality also depends on:

```text
training data
training objective
architecture
model capacity
representation strategy
```

Dimension is only one knob.